# 06 Grouped SKU Demo — результат склейки на DuckDB

Эта тетрадка показывает не метрики, а сам результат: берём реальные SKU из `mpstats_products` в DuckDB, накладываем на них текущие `fusion_family_id` и `fusion_pack_id` из `04_fusion_pack_grouping.ipynb`, затем смотрим, какие карточки реально попали в одну группу.

Важно: это демонстрация текущего research-среза. Она показывает SKU, которые попали в `fusion_components_<suffix>.csv` текущего category-run, а не production-dedup всего куба.


## Что здесь считается результатом

- `fusion_family_id` — базовый товар: разные карточки одного и того же продукта.
- `fusion_pack_id` — конкретная фасовка внутри family.
- DuckDB нужен, чтобы показать реальные строки куба: маркетплейс, артикул, название, продажи, выручку и месяцы.


## Блок кода 1. Подготовка окружения


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.width", 220)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "research" / "dedup").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.dedup import resolve_category_run, resolve_run_paths

PROJECT_ROOT


## Блок кода 2. Где лежат DuckDB и research-артефакты


In [ ]:

CATEGORY_RUN = resolve_category_run()
RUN_PATHS = resolve_run_paths(PROJECT_ROOT, CATEGORY_RUN)
DATA_DIR = RUN_PATHS.data_dir
REPORTS_DIR = RUN_PATHS.reports_dir

COMPONENTS_PATH = RUN_PATHS.fusion_components_path
PAIR_EVAL_PATH = RUN_PATHS.fusion_pair_eval_path

DB_CANDIDATES = [
    os.environ.get("MPSTATS_DUCKDB_PATH"),
    PROJECT_ROOT / "mpstats.duckdb",
    Path("/Users/exoldoff/Desktop/mpstats/mpstats.duckdb"),
]
DB_CANDIDATES = [Path(path) for path in DB_CANDIDATES if path]
DUCKDB_PATH = next((path for path in DB_CANDIDATES if path.exists()), None)

missing = [path for path in [COMPONENTS_PATH, PAIR_EVAL_PATH] if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Сначала запустите notebooks/04_fusion_pack_grouping.ipynb для category-run "
        f"{CATEGORY_RUN.slug!r}: не хватает "
        + ", ".join(str(path) for path in missing)
    )
if DUCKDB_PATH is None:
    raise FileNotFoundError(
        "Не нашёл mpstats.duckdb. Задайте MPSTATS_DUCKDB_PATH или положите базу в корень проекта."
    )

print(f"Category run: {CATEGORY_RUN.slug} — {CATEGORY_RUN.display_name}; project: {CATEGORY_RUN.project_name}")
print(f"DuckDB: {DUCKDB_PATH}")
print(f"Fusion components: {COMPONENTS_PATH}")
print(f"Fusion pair eval: {PAIR_EVAL_PATH}")


## Блок кода 3. Загружаем текущие dedup-группы


In [ ]:
components = pd.read_csv(COMPONENTS_PATH)
pair_eval = pd.read_csv(PAIR_EVAL_PATH)

run_cols = ["fusion_method", "fusion_threshold_strategy", "fusion_threshold_same"]
run_info = components[run_cols].drop_duplicates() if set(run_cols).issubset(components.columns) else pd.DataFrame()

component_summary = pd.DataFrame(
    [
        {"metric": "fusion_sku_rows", "value": len(components)},
        {"metric": "unique_nodes", "value": components["node_id"].nunique()},
        {"metric": "family_groups", "value": components["fusion_family_id"].nunique()},
        {"metric": "pack_groups", "value": components["fusion_pack_id"].nunique()},
    ]
)

display(run_info)
display(component_summary)
display(components.head(10))


## Блок кода 4. Читаем реальные SKU из DuckDB


In [ ]:

import duckdb

CATEGORY_ALIASES = list(CATEGORY_RUN.category_aliases)
PROJECT_NAME = CATEGORY_RUN.project_name
TABLE_NAME = "mpstats_products"

with duckdb.connect(str(DUCKDB_PATH), read_only=True) as con:
    table_info = con.execute(f"PRAGMA table_info('{TABLE_NAME}')").fetchdf()
    available_columns = set(table_info["name"])
    where_clauses = []
    params: list[object] = []
    if "Категория" in available_columns:
        placeholders = ", ".join(["?"] * len(CATEGORY_ALIASES))
        where_clauses.append(f'"Категория" IN ({placeholders})')
        params.extend(CATEGORY_ALIASES)
    if PROJECT_NAME:
        if "__project_name" not in available_columns:
            raise RuntimeError("Для category-run нужен project-фильтр, но в кубе нет __project_name.")
        where_clauses.append('"__project_name" = ?')
        params.append(PROJECT_NAME)
    where_sql = f" WHERE {' AND '.join(where_clauses)}" if where_clauses else ""
    products = con.execute(f'SELECT * FROM "{TABLE_NAME}"{where_sql}', params).fetchdf()

print(f"DuckDB rows for {PROJECT_NAME!r} / {CATEGORY_ALIASES}: {len(products):,}")
print(f"DuckDB unique columns: {len(available_columns)}")
display(table_info[["name", "type"]])
display(products.head(5))


## Блок кода 5. Собираем витрину SKU из DuckDB


In [ ]:
def _clean_article(value: object) -> str:
    if pd.isna(value):
        return ""
    text = str(value).strip()
    return text[:-2] if text.endswith(".0") else text


def _norm_marketplace(value: object) -> str:
    if pd.isna(value):
        return ""
    return str(value).strip().lower()


def _first_nonempty(values: pd.Series) -> str | None:
    clean = values.dropna().astype(str).str.strip()
    clean = clean[clean.ne("")]
    return clean.iloc[0] if not clean.empty else None


def _joined_unique(values: pd.Series, limit: int = 12) -> str:
    clean = sorted(set(values.dropna().astype(str).str.strip()) - {""})
    suffix = "" if len(clean) <= limit else f" ... +{len(clean) - limit}"
    return ", ".join(clean[:limit]) + suffix


products = products.copy()
products["node_id"] = products["Маркетплейс"].map(_norm_marketplace) + "::" + products["Артикул"].map(_clean_article)

sales_col = "Продажи, шт" if "Продажи, шт" in products.columns else None
revenue_col = "Выручка, руб" if "Выручка, руб" in products.columns else None
price_col = "Средняя цена, руб" if "Средняя цена, руб" in products.columns else None
unit_col = "Вес, кг (ед.)" if "Вес, кг (ед.)" in products.columns else None
total_col = "Вес, кг" if "Вес, кг" in products.columns else None

rows: list[dict[str, object]] = []
for node_id, group in products.groupby("node_id", dropna=False):
    if sales_col:
        sales_values = group[sales_col].fillna(0)
        representative = group.loc[sales_values.idxmax()]
    else:
        representative = group.iloc[0]

    unit_amount = representative[unit_col] if unit_col else None
    total_amount = representative[total_col] if total_col else None
    multipack_count = None
    if pd.notna(unit_amount) and pd.notna(total_amount) and float(unit_amount) > 0:
        multipack_count = round(float(total_amount) / float(unit_amount))

    sales_qty = float(group[sales_col].fillna(0).sum()) if sales_col else None
    revenue_rub = float(group[revenue_col].fillna(0).sum()) if revenue_col else None
    avg_price = (revenue_rub / sales_qty) if sales_qty and revenue_rub is not None else None

    rows.append(
        {
            "node_id": node_id,
            "db_marketplace": representative.get("Маркетплейс"),
            "db_sku": _clean_article(representative.get("Артикул")),
            "db_title": representative.get("SKU"),
            "db_brand": representative.get("Бренд"),
            "db_category": representative.get("Категория"),
            "db_unit_amount": unit_amount,
            "db_total_amount": total_amount,
            "db_multipack_count": multipack_count,
            "db_sales_qty": sales_qty,
            "db_revenue_rub": revenue_rub,
            "db_avg_price_rub": avg_price,
            "db_rows": len(group),
            "db_projects": _joined_unique(group["__project_name"]) if "__project_name" in group.columns else None,
            "db_months": _joined_unique(group["месяц"]) if "месяц" in group.columns else None,
        }
    )

db_sku_catalog = pd.DataFrame(rows)

catalog_summary = pd.DataFrame(
    [
        {"metric": "duckdb_sauce_rows", "value": len(products)},
        {"metric": "duckdb_unique_sku_nodes", "value": db_sku_catalog["node_id"].nunique()},
    ]
)
display(catalog_summary)
display(db_sku_catalog.head(10))


## Блок кода 6. Накладываем dedup-группы на DuckDB SKU


In [ ]:
joined = components.merge(db_sku_catalog, on="node_id", how="left", validate="one_to_one")
joined["title_demo"] = joined["db_title"].combine_first(joined.get("title"))
joined["brand_demo"] = joined["db_brand"].combine_first(joined.get("brand"))
joined["marketplace_demo"] = joined["db_marketplace"].combine_first(joined.get("marketplace"))
joined["sku_demo"] = joined["db_sku"].combine_first(joined.get("sku").astype(str))
joined["unit_amount_demo"] = joined["db_unit_amount"].combine_first(joined.get("unit_amount"))
joined["total_amount_demo"] = joined["db_total_amount"].combine_first(joined.get("total_amount"))
joined["multipack_count_demo"] = joined["db_multipack_count"].combine_first(joined.get("multipack_count"))
joined["sales_qty_demo"] = joined["db_sales_qty"].combine_first(joined.get("sales_volume"))

coverage = pd.DataFrame(
    [
        {"metric": "fusion_nodes", "value": joined["node_id"].nunique()},
        {"metric": "matched_in_duckdb", "value": int(joined["db_title"].notna().sum())},
        {"metric": "missing_in_duckdb", "value": int(joined["db_title"].isna().sum())},
        {"metric": "family_groups", "value": joined["fusion_family_id"].nunique()},
        {"metric": "multi_sku_family_groups", "value": int((joined.groupby("fusion_family_id")["node_id"].nunique() > 1).sum())},
        {"metric": "multi_sku_pack_groups", "value": int((joined.groupby("fusion_pack_id")["node_id"].nunique() > 1).sum())},
    ]
)
display(coverage)

missing = joined[joined["db_title"].isna()][["node_id", "marketplace", "sku", "title"]]
if not missing.empty:
    print("Не найдено в DuckDB:")
    display(missing.head(20))


## Блок кода 7. Какие family-группы реально склеились


In [ ]:
def _unique_join(values: pd.Series, limit: int = 5) -> str:
    clean = sorted(set(values.dropna().astype(str).str.strip()) - {""})
    suffix = "" if len(clean) <= limit else f" ... +{len(clean) - limit}"
    return ", ".join(clean[:limit]) + suffix


family_rows: list[dict[str, object]] = []
for family_id, group in joined.groupby("fusion_family_id", dropna=False):
    rep_idx = group["sales_qty_demo"].fillna(0).idxmax() if "sales_qty_demo" in group else group.index[0]
    rep = group.loc[rep_idx]
    family_rows.append(
        {
            "fusion_family_id": family_id,
            "sku_count": group["node_id"].nunique(),
            "pack_group_count": group["fusion_pack_id"].nunique(),
            "marketplaces": _unique_join(group["marketplace_demo"]),
            "brands": _unique_join(group["brand_demo"]),
            "total_sales_qty": group["sales_qty_demo"].fillna(0).sum(),
            "total_revenue_rub": group["db_revenue_rub"].fillna(0).sum(),
            "representative_title": rep["title_demo"],
        }
    )

family_summary = pd.DataFrame(family_rows)
glued_families = family_summary[family_summary["sku_count"] > 1].sort_values(
    ["sku_count", "total_sales_qty"], ascending=[False, False]
)

display(glued_families.head(30))

try:
    import matplotlib.pyplot as plt

    size_counts = glued_families["sku_count"].value_counts().sort_index()
    ax = size_counts.plot(kind="bar", figsize=(8, 3), color="#3B82F6")
    ax.set_title("Сколько SKU в склеенных family-группах")
    ax.set_xlabel("SKU в family")
    ax.set_ylabel("Количество family")
    plt.tight_layout()
    plt.show()
except Exception as exc:
    print(f"График пропущен: {exc}")


## Блок кода 8. Посмотреть конкретную склеенную family


In [ ]:
SHOW_FAMILY_ID = None  # пример: 29. None = взять самую крупную/продаваемую группу

if glued_families.empty:
    print("Нет family-групп с несколькими SKU")
else:
    selected_family_id = SHOW_FAMILY_ID if SHOW_FAMILY_ID is not None else glued_families.iloc[0]["fusion_family_id"]
    selected_family = joined[joined["fusion_family_id"].eq(selected_family_id)].copy()
    selected_family = selected_family.sort_values(["fusion_pack_id", "sales_qty_demo", "marketplace_demo"], ascending=[True, False, True])

    detail_cols = [
        "fusion_family_id",
        "fusion_pack_id",
        "marketplace_demo",
        "sku_demo",
        "title_demo",
        "brand_demo",
        "unit_amount_demo",
        "total_amount_demo",
        "multipack_count_demo",
        "sales_qty_demo",
        "db_revenue_rub",
        "db_months",
        "node_id",
    ]
    print(f"Selected fusion_family_id: {selected_family_id}")
    display(selected_family[detail_cols])


## Блок кода 9. Pack-группы: где совпала конкретная фасовка


In [ ]:
pack_rows: list[dict[str, object]] = []
for (family_id, pack_id), group in joined.groupby(["fusion_family_id", "fusion_pack_id"], dropna=False):
    if group["node_id"].nunique() <= 1:
        continue
    rep_idx = group["sales_qty_demo"].fillna(0).idxmax()
    rep = group.loc[rep_idx]
    pack_rows.append(
        {
            "fusion_family_id": family_id,
            "fusion_pack_id": pack_id,
            "sku_count": group["node_id"].nunique(),
            "marketplaces": _unique_join(group["marketplace_demo"]),
            "total_sales_qty": group["sales_qty_demo"].fillna(0).sum(),
            "unit_amount": rep["unit_amount_demo"],
            "total_amount": rep["total_amount_demo"],
            "multipack_count": rep["multipack_count_demo"],
            "representative_title": rep["title_demo"],
        }
    )

pack_summary = pd.DataFrame(pack_rows).sort_values(["sku_count", "total_sales_qty"], ascending=[False, False])
display(pack_summary.head(30))


## Блок кода 10. Все склеенные SKU одной таблицей


In [ ]:
family_size = joined.groupby("fusion_family_id")["node_id"].transform("nunique")
grouped_skus = joined[family_size > 1].copy()
grouped_skus["family_sku_count"] = family_size[family_size > 1].values

demo_cols = [
    "fusion_family_id",
    "family_sku_count",
    "fusion_pack_id",
    "marketplace_demo",
    "sku_demo",
    "title_demo",
    "brand_demo",
    "unit_amount_demo",
    "total_amount_demo",
    "multipack_count_demo",
    "sales_qty_demo",
    "db_revenue_rub",
    "db_months",
    "node_id",
]
grouped_skus = grouped_skus.sort_values(["fusion_family_id", "fusion_pack_id", "sales_qty_demo"], ascending=[True, True, False])

display(grouped_skus[demo_cols].head(200))

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_PATH = RUN_PATHS.grouped_sku_demo_path
grouped_skus[demo_cols].to_csv(EXPORT_PATH, index=False)
print(f"Saved demo CSV: {EXPORT_PATH} ({len(grouped_skus)} rows)")


## Блок кода 11. Какие pair-edges склеили выбранную family


In [ ]:
def _as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.strip().str.lower().isin({"true", "1", "yes"})


if glued_families.empty:
    print("Нет склеенных family-групп")
else:
    selected_family_id = SHOW_FAMILY_ID if SHOW_FAMILY_ID is not None else glued_families.iloc[0]["fusion_family_id"]
    edges = pair_eval[_as_bool(pair_eval["fusion_family_edge"])].copy()
    if {"fusion_family_id_a", "fusion_family_id_b"}.issubset(edges.columns):
        edges = edges[
            edges["fusion_family_id_a"].eq(selected_family_id)
            | edges["fusion_family_id_b"].eq(selected_family_id)
        ]

    edge_cols = [
        "split",
        "score",
        "fusion_threshold_same",
        "fusion_pack_edge",
        "same_pack_signature",
        "marketplace_a",
        "sku_a",
        "title_a",
        "marketplace_b",
        "sku_b",
        "title_b",
    ]
    edge_cols = [column for column in edge_cols if column in edges.columns]
    print(f"Positive pair edges for fusion_family_id={selected_family_id}")
    display(edges[edge_cols].sort_values("score", ascending=False).head(30))


## Как пользоваться

1. Сначала прогоните `03_matching_comparison.ipynb`, потом `04_fusion_pack_grouping.ipynb`.
2. Запустите эту тетрадку сверху вниз.
3. В блоке `SHOW_FAMILY_ID` можно подставить любой `fusion_family_id` из таблицы склеенных family.
4. CSV с витриной склеенных SKU сохраняется в `artifacts/reports/dedup_grouped_sku_demo.csv`.

Если нужна склейка всего DuckDB, а не только research-среза из `fusion_components_sauces.csv`, следующий шаг — отдельный production/research job, который прогонит candidate generation + scorer + fusion по всему набору SKU и сохранит identity mapping.
